In [1]:
%matplotlib inline

# Board Game Simulation

## Volcano Rush

### Abstract

This study uses Monte Carlo simulation with rule-based agents to evaluate the balance of *Volcano Rush*, a semi-cooperative board game for 6-8 players. Thousands of games (1,000-10,000 per scenario) are simulated across player counts of 6, 7, and 8, recording win rates, individual scores by character role, and resource consumption patterns.

The simulations show that 6-player (51.9% win rate) and 7-player (56.4%) configurations fall within the target 50-65% range, while 8-player games are too easy at 75.0% - the 5th boat part does not offset the extra resources and actions available to larger groups. Character balance reveals a structural split: Cook and Fire Starter achieve personal win rates of 19.0% and 18.0% respectively (above the 16.7% uniform baseline), while Sailor (12.7%) and Thief (13.9%) underperform because the scoring system does not capture their support contributions. Resource analysis identifies Rope as the tightest bottleneck (2.64 mission failures per game) and Stone as underused (0.28 failures per game), with the extra-resource rule having negligible impact.

These results suggest three design adjustments: rebalancing 8-player difficulty, introducing scoring mechanisms for support roles, and redistributing resource requirements across missions.

### Introduction

Game balance is considered critical to the success of any game, yet there is no universal consensus on what it actually means [1]. It can be broadly interpreted as the process of tuning a game's rules, difficulty, play time, resources, and role abilities so that no single strategy, character, or configuration dominates the experience. For multiplayer games in particular, balance has several dimensions: ensuring no starting position is inherently advantageous, preventing any strategy from being strictly dominant, and calibrating the cost-to-benefit ratio across different game objects [1].

As a semi-cooperative board game, *Volcano Rush* faces a specific balancing challenge: the group must share a win condition (escaping the volcano), while players also compete for individual scores. A well-balanced game therefore needs a group win rate that feels achievable but not trivial, and individual scores that are not dominated by a single character role. Compounding this, the game supports 4 to 8 players, meaning balance must hold across a wide range of player configurations.

As with most game design problems, there is no deterministic recipe for achieving balance. *Volcano Rush* is both strategic and social, and raw numbers can sometimes mislead. Simulation is not a substitute for playtesting, but it provides a principled starting point for identifying where imbalances may arise before bringing the game to the table.

### Related Work

#### Monte Carlo simulation

Monte Carlo simulation estimates probabilities by running many random trials instead of solving mathematical formulas, which can be extremely difficult for complex games. Running $n$ simulated games under a given strategy $M$ yields an empirical win probability:

We define an indicator variable for each simulation:

* $X_i = 1$ if simulation $i$ results in a win  
* $X_i = 0$ otherwise  

The empirical win probability after $n$ simulations is:

$$
\hat{p}_n(M) = \frac{1}{n} \sum_{i=1}^{n} X_i
$$

As the number of simulations increases, this estimate converges to the true win probability:

$$
p(M) = \lim_{n \to \infty} \frac{1}{n} \sum_{i=1}^{n} X_i
$$

This follows from the **Law of Large Numbers**, which states that the sample average converges to the expected value (true probability) as the number of trials grows. This makes Monte Carlo methods a good fit for games with many possible situations, where computing the exact answer is not practical.

#### Monte Carlo Tree Search (MCTS)

Monte Carlo methods are also widely used in game AI through Monte Carlo Tree Search (MCTS) [2], which builds a tree of possible future game states and tests them with simulations. 
At each step, the algorithm faces a trade-off between two ideas:
* *exploitation* - choosing moves that have already shown good results  
* *exploration* - trying moves that have not been tested much but might be better  

This balance is handled by the **Upper Confidence Bound for Trees** (UCT) formula:

$$ \mathrm{UCT} = \frac{W_i}{N_i} + c \sqrt{\frac{\ln N_p}{N_i}} $$

The formula combines two parts:
* $\frac{W_i}{N_i}$ - the **exploitation term**, which represents how successful a move has been so far (its average win rate)
* $c \sqrt{\frac{\ln N_p}{N_i}}$ - the **exploration term**, which gives a higher value to moves that have been tried fewer times, encouraging the algorithm to explore them
* $c$ - a constant that controls the balance between exploration and exploitation (higher values lead to more exploration)

In practice, MCTS does not explore all possible game paths. Instead, it gradually builds the search tree through repeated simulations. At the beginning, it gathers initial data by trying different possible moves. As more simulations are performed, the algorithm starts to focus more on the moves that show better results, exploring them in greater depth. However, less-explored moves are still occasionally tested due to the exploration term in the UCT formula, ensuring that potentially good alternatives are not missed.

In this work, however, we use simpler repeated simulations with rule-based agents to study game balance rather than to improve decision-making [2].

#### Agent-Based Modeling (ABM)

In Agent-Based Modeling (ABM), each agent follows a set of simple rules and interacts with other agents and the world around it [3]. Over time, these small interactions can produce complex group behaviors that would be hard to predict just by looking at the rules alone.

Compared to pure Monte Carlo simulation, ABM allows for more realistic behavior. Monte Carlo is fully random, but in ABM, agents can follow rules that reflect real human decisions. This approach has been used in research - for example, interviewing participants after a game experiment and using their answers to design the decision rules for ABM agents [5]. For *Volcano Rush*, we could do the same - interview players after a playtest session and use what we learn to make the agents behave more like real people [4].

Traditional game theory assumes that players always make the best possible decision to get the best outcome for themselves. But in real life, people are often influenced by emotions, habits, and not having all the information [3]. This means game theory models do not always reflect how real people actually play.

### Data

This study uses no external dataset. All data is generated by the simulation itself - each run of the simulation produces one game record. The input to the simulation is a fixed set of game parameters taken directly from the game rules.

#### Game Parameters

The resource deck contains **60 cards** split equally between three types: Wood, Stone, and Rope (20 each). The deck is reshuffled and reused when it runs out. The camp has two **shared tools** - Knife and Vessel - which can become damaged and must be repaired before use.

There are **6 character roles**: Builder, Fire Starter, Craftsman, Cook, Thief, and Sailor. Each has a unique ability that affects resource requirements, point rewards, or complication handling. For games of 4 or 5 players, the Craftsman role is always assigned and the remaining roles are drawn randomly. For 6 players, all six roles are used once. For 7-8 players, one or two roles are repeated.

The **mission pool** consists of 8 standard missions and up to 5 boat missions. At any time, exactly 3 missions are active and new missions are drawn when one is completed or discarded. The number of boat missions in play - and the number of boat parts needed to win - depends on player count:

| Players | Boat Parts Required | Active Boat Missions |
|---|---|---|
| 4-5 | 2 | 2 randomly selected |
| 6 | 4 | All 4 base missions |
| 7-8 | 5 | All 4 base missions + Fit the Rudder |

Boat parts must be completed in a fixed construction order: Cut the Keel -> Assemble the Hull -> Raise the Mast -> Make the Sail -> Fit the Rudder.

The **Complication deck** contains 11 cards (including 2 neutral cards with no effect) and is reshuffled when exhausted. The **Volcano deck** contains 11 penalty cards with the Eruption card fixed at the bottom - when it is drawn, the game ends in a loss.
Card draws from the resource, complication, and volcano decks are modeled as stochastic processes with uniform random sampling.

#### Simulation Assumptions

Where the game rules leave room for interpretation, the following assumptions are applied:

| Rule Area | Assumption |
|---|---|
| Active player | Rotates through the player list each round; the active player decides the round's action (mission, shuffle, or fallback volcano draw) |
| Mission selection | The active player selects the mission unilaterally using their character's voting preference |
| Participant selection | The active player selects participants in two pools: preferred (non-exhausted, ≥2 resources) and fallback (non-exhausted, 1 resource). Players with 0 resources or exhausted players are excluded |
| Resource payment | Resources are collected from constrained players first (fewer resources), then from flexible players (more resources). This approximates real-game communication where the group optimizes who pays what |
| Participation cost | Every participant must pay at least 1 resource. Players who were not charged during normal resource deduction pay 1 resource of any type |
| Shuffle decision | If no boat parts are visible in active missions, the active player randomly chooses between shuffling and picking a mission; if all active missions are boat parts but the next needed part is missing, shuffling is mandatory |
| Shuffle cost | Costs the active player 1 resource card of any type; cannot shuffle with 0 resources |
| Exhaustion | Applies to all participants after resolution, regardless of success or failure |
| Craftsman repair | Automatic when not participating: if not exhausted, has Stone, and a tool is damaged with no repair in progress. Starts round N, tool unavailable round N+1, available again round N+2. Craftsman does not gather when repairing |
| Thief steal | Automatic when not participating: if not exhausted and holds fewer than 3 resources, steals 1 random resource from each of 2 different players with at least 1 resource, then becomes Exhausted. Falls back to normal gathering if fewer than 2 valid targets exist |
| Sailor - lesser evil | The less severe Complication card is chosen using a severity scoring function |
| Participant count | Treated as an exact requirement, no additional participants allowed |
| Damaged tool | A mission requiring a damaged tool automatically fails |
| Night Anxiety | Requires 1 non-participant helper; if none available, mission fails |
| Gather | Any non-participant who did not use a special ability (Craftsman repair, Thief steal) draws 1 resource from the deck. Drawing 1 resource does not cause Exhaustion |

#### Output Variables

Each simulation run records: game outcome (win/loss), number of rounds played, individual score per player, win rate per role (role dominance), number of boat parts completed, and number of volcano cards remaining when the game ends.

### Methods

The simulation runs each game as a sequence of rounds following the round structure described in `docs/game_rules.md`. Thousands of games are run per configuration, and outcomes are recorded to estimate win rates and individual scores empirically.

#### Agent Strategy

Each round is led by a rotating **active player** who makes key decisions. The remaining players follow simple rules based on whether they are selected as participants.

**Active player decision (start of each round)**
The active player chooses one of the following actions:
1. **Choose a mission** - select one of the 3 active missions using their character preference (e.g. Builder prefers Wood-heavy missions, Cook prefers Vessel missions).
2. **Shuffle the mission deck** - reshuffle all undrawn missions. Costs 1 resource card from the active player. No mission is attempted and no gathering occurs this round.

The decision follows these rules in priority order:
* If all 3 active missions are boat parts AND the next needed boat part (by construction order) is not among them → **shuffle** (mandatory).
* If none of the 3 active missions are boat parts → randomly choose between **shuffle** and **choose a mission**.
* If the active player has no resources → **choose a mission** (cannot afford to shuffle).
* Otherwise → **choose a mission**.

**Participant selection (when a mission is chosen)**
The active player selects participants for the mission:
* Players with **≥2 resources** are preferred and selected first (randomly within this group).
* Players with **1 resource** are a fallback option.
* Players with **0 resources** or **Exhausted** players are excluded.

**Non-participant actions**
* **Craftsman** (if not Exhausted, has Stone, and a tool is damaged with no repair in progress): automatically repairs a tool (+1 point), skips gathering.
* **Thief** (if not Exhausted and holds fewer than 3 cards): steals 1 resource from each of 2 random players who have at least 1 resource; becomes Exhausted afterward. If fewer than 2 valid targets exist, gathers normally instead.
* All other non-participants: **Gather** - draw 1 random resource card.

**Boat mission priority**
* When the volcano deck has 4 or fewer cards remaining, the active player gives maximum priority to boat missions in their mission selection.

#### Simulation Scenarios

Games are run across all player counts (4-8). Character assignment follows the game rules: Craftsman is always assigned in 4-5 player games; all 6 roles are used exactly once in 6-player games; repeated roles fill the extra slots in 7-8 player games. Each scenario runs N = 1,000-10,000 games. Sensitivity analysis varies the participation threshold and boat mission priority trigger.

#### Statistical Analysis

Win rates per scenario are estimated empirically and reported with 95% Wald confidence intervals.

#### Simulation Engine

The engine exposes two public functions:

* `run_game(player_count, ...)` - runs a single game and returns a `GameRecord`
* `run_scenario(player_count, n_games, base_seed, ...)` - runs `n_games` games and returns a list of `GameRecord`. When `base_seed` is provided, each game is seeded as `base_seed + i`, making the entire scenario reproducible.

`GameRecord` captures the outcome, number of rounds played, character list, final scores per character, boat parts built vs. required, and volcano cards remaining at game end.

Each game runs as a sequence of rounds via `run_round`, which executes up to 10 steps in order:

**active player decision → mission selection → participant selection → non-participant actions → complication draw → mission resolution → exhaustion → gather → mission refresh → advance active player**

If the active player chooses to shuffle the mission deck, the round ends immediately after the shuffle cost is paid and the deck is reshuffled - steps 2-9 are skipped entirely.

#### Agent Heuristics

Each round is controlled by three agent functions that encode hand-coded decision rules.

**Active player decision - `decide_active_player_action`**

The active player chooses between `CHOOSE_MISSION` and `SHUFFLE_MISSIONS`:

1. All active missions are boat parts AND the next needed boat part (by construction order) is missing from the active missions → **SHUFFLE_MISSIONS** (mandatory)
2. No boat parts visible among active missions → random choice between **SHUFFLE_MISSIONS** and **CHOOSE_MISSION**
3. Otherwise → **CHOOSE_MISSION**

If **SHUFFLE_MISSIONS** would be chosen but the active player has no resources, the action falls back to **CHOOSE_MISSION** (shuffle requires 1 resource as payment).

**Mission selection - `vote_for_mission`**

The active player selects a mission using their character preference:

* If the volcano deck has ≤ `urgent_volcano_threshold` cards remaining (default 4), the active player ignores their character preference and selects a boat mission.
* Otherwise, each character has a preference: Builder selects missions requiring ≥2 Wood, Fire Starter selects fire-type missions, Cook selects missions requiring the Vessel tool, Sailor selects boat missions. Craftsman and Thief have no preference and select randomly.

**Participant selection - `active_player_select_participants`**

The active player selects up to `mission.players_count` participants:

* Pool 1 (preferred): non-exhausted players with ≥2 resources - shuffled and drawn first.
* Pool 2 (fallback): non-exhausted players with exactly 1 resource - drawn only if Pool 1 is insufficient.
* Players with 0 resources or exhausted players are excluded entirely.

**Non-participant actions - `take_gathering_action`**

Each non-participant's character strategy decides what to do. If `take_gathering_action` returns `True`, the player gathers from the resource deck. If it returns `False`, the player used a special ability instead (Craftsman repairs a tool, Thief steals from other players) and skips the deck draw.

* **Craftsman**: if not Exhausted, has Stone, and a tool is damaged with no repair in progress, repairs the tool (+1 point) and skips gathering.
* **Thief**: if not Exhausted and holds fewer than 3 resources, steals 1 random resource from each of 2 different players who have at least 1 resource. Becomes Exhausted afterward. Falls back to normal gathering if fewer than 2 valid targets exist.
* **All others**: gather 1 resource from the deck.

#### Simulations

* See [1_simulation_verbose_run.ipynb](1_simulation_verbose_run.ipynb).
* See [2_simulation_player_count_balance.ipynb](2_simulation_player_count_balance.ipynb).
* See [3_simulation_character_balance.ipynb](3_simulation_character_balance.ipynb).
* See [4_simulation_resource_efficiency.ipynb](4_simulation_resource_efficiency.ipynb).

### Discussion

The simulations reveal several balance issues in the current design:

* **8-player balance is the main problem.** The win rate (75.0%) is well above the 50-65% target. The 5th boat part doesn't add enough difficulty to offset the extra resources and actions from more players. This directly reflects an imbalanced cost-to-benefit ratio across player counts [1].
* **Character balance shows a clear split.** Cook, Craftsman, Fire Starter, and Builder cluster near the 16.7% baseline for personal win rate, while Sailor (12.7%) and Thief (13.9%) fall behind. The scoring system only rewards mission completion points, missing the hidden contributions these characters make (Sailor's complication mitigation, Craftsman's tool repair).
* **Volcano tension weakens with more players.** For 6 players the median is 3 cards remaining at the time of a win, which is acceptable. For 8 players it rises to 6, meaning larger groups rarely feel the eruption threat.
* **Rope is the tightest resource bottleneck** despite similar consumption to Wood. Stone is underused. The extra resource rule has almost no effect on mission outcomes.

### Limitations

* **Fixed strategies.** Agents use the same heuristics every game, so the simulation cannot measure strategy diversity - how many different paths to victory exist. A well-balanced game should have no strictly dominant strategy [1].
* **No communication model.** Real players negotiate who pays what, which missions to prioritize, and when to shuffle. Traditional game theory assumes optimal decision-making, but real players are influenced by emotions, habits, and incomplete information [3]. This social layer can shift balance in ways the simulation can't capture. Interviewing players after playtests and using their answers to design agent decision rules [4][5] would produce more realistic behavior.
* **Incomplete scoring visibility.** Characters with support roles (Sailor, Craftsman) are undervalued because their contributions are invisible to the current mission-point-only metrics. A hidden points system that tracks group and personal effectiveness would give a more complete picture.

### Further Work

* **First-player advantage.** The starting active player makes the first mission decision and scores first. Over many rounds the rotation evens out, but early-round advantage in resource spending and point accumulation could affect personal win rate. A balanced game should ensure no starting position is inherently advantageous [1]. A simulation comparing average score by player position (1st through 6th) would reveal whether starting order matters.
* **Strategy diversity (hardest).** Test whether different approaches lead to different outcomes. This requires either multiple distinct agent types or parameterized behavior weights (e.g., aggressive vs. conservative mission selection). Also test the urgent volcano flag at different thresholds.
* **Volcano card effects.** Track which card was in play when losses happen and which characters help survive specific effects. Some cards compound over time (e.g., Ash in the Air stacking exhaustion) and won't show as the last card before a loss but are still impactful - this needs cumulative card tracking per game.
* **Complication card effects.** Same loss-correlation approach. Most interesting: test modified decks with harsher effects repeated (e.g., double the Storm cards) and compare win rates. The Sailor's ability (draw 2 complications, keep the less severe one) is directly testable here.
* **Rule variant testing.** Test with different resource requirements, harsher volcano/complication decks, and adjusted boat part counts to find better balance for 8 players.
* **MCTS agents.** Replace rule-based heuristics with Monte Carlo Tree Search [2] for more realistic decision-making. Adds complexity but would produce more reliable balance estimates.
* **Engine tracking extension.** The volcano and complication card simulations require the engine to record which cards appeared and when. GameRecord needs to be extended with card event tracking before those simulations can run.

### References / Bibliography

[1] [Schell, J. (2009). Level 16: Game Balance. Game Design Concepts.](https://gamedesignconcepts.wordpress.com/2009/08/20/level-16-game-balance/)

[2] [Chaslot, G., Bakkes, S., Szita, I., & Spronck, P. (2008). Monte-Carlo Tree Search: A New Framework for Game AI. AIIDE.](https://sander.landofsand.com/publications/AIIDE08_Chaslot.pdf)

[3] [StudyGuides.com. (2026). Simulations in Game Theory.](https://studyguides.com/study-methods/overview/clzggcm6j0j8t8cfe9zugo2t8)

[4] [Dubois, E., Barreteau, O., & Souchère, V. (2013). An Agent-Based Model to Explore Game Setting Effects on Attitude Change During a Role Playing Game Session. Journal of Artificial Societies and Social Simulation, 16(1), 2.](https://www.jasss.org/16/1/2.html)

[5] [Yiannakoulias, N., Grignon, M., & Marshall, T. (2024). Parameterizing agent-based models using an online game. Computers, Environment and Urban Systems, 112, 102142.](https://www.sciencedirect.com/science/article/pii/S0198971524000711)

### Appendix

* **Appendix A:** General game rules (overview, resources, mechanics, scoring) - `docs/game_rules.md`
* **Appendix B:** Mission table - `docs/missions.md`
* **Appendix C:** Complication card descriptions - `docs/complication_cards.md`
* **Appendix D:** Volcano card descriptions - `docs/volcano_cards.md`
* **Appendix E:** Character ability reference - `docs/characters.md`